# AssistIQ — Phase 2: Historical Support Retrieval / Knowledge Base

This notebook provides an interactive demonstration of **Phase 2: Historical Support Retrieval** for **AssistIQ** (@SpotifyCares Twitter Customer Support).

### Pipeline Flow
```
Customer message
      ↓
Dense Semantic Embedding (all-MiniLM-L6-v2, 384-dim, L2-normalized)
      ↓
FAISS Vector Search (IndexFlatIP: Cosine Similarity)
      ↓
Top-K Similar Historical Customer Cases
      ↓
Associated Historical Spotify Responses / Evidence
```

### Key Sections
1. **Historical Corpus Statistics**: Transformation audit trail from raw tweets to clean cases.
2. **Example Customer / Support Pairs**: Realistic paired support interactions.
3. **Embedding Model**: Model architecture and dimensional specifications.
4. **Example Embeddings & Cosine Similarity**: Pairwise semantic distance matrix.
5. **FAISS Index**: Vector index structure, persistence, and loading.
6. **Example Semantic Searches**: Interactive queries across support domains.
7. **Top-K Retrieved Cases**: Granular metadata inspection.
8. **Similarity Scores**: Distribution analysis.
9. **Qualitative Retrieval Examples**: Semantic vs. lexical matches and failure cases.

### 1. Import Dependencies & Path Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root and backend are in sys.path
project_root = os.path.abspath(os.path.join(".."))
backend_dir = os.path.join(project_root, "backend")
for p in [project_root, backend_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

from backend.src.retrieval.data import load_support_cases
from backend.src.retrieval.embeddings import SentenceEmbeddingModel, clean_for_embedding
from backend.src.retrieval.index import FAISSRetrievalIndex, get_default_artifacts_dir
from backend.src.retrieval.retrieve import retrieve_similar_cases

### 2. Historical Corpus Statistics & Transformation Audit Trail
We load `dataset/spotify_support_cases.csv` and inspect the dataset transformation.

In [ ]:
corpus_path = os.path.join(project_root, "dataset", "spotify_support_cases.csv")
df_cases = load_support_cases(corpus_path)

print("==================================================================")
print("HISTORICAL RETRIEVAL CORPUS AUDIT")
print("==================================================================")
print(f"• Total Historical Cases:       {len(df_cases):,}")
print(f"• Unique Customer Messages:     {df_cases['customer_text'].nunique():,}")
print(f"• Unique Support Responses:     {df_cases['support_text'].nunique():,}")
print(f"• Unique Conversation Threads:  {df_cases['conversation_id'].nunique():,}")
print(f"• Missing / Null Customer Text: {df_cases['customer_text'].isna().sum()}")
print(f"• Missing / Null Support Text:  {df_cases['support_text'].isna().sum()}")
print("==================================================================")
df_cases.head(3)

### 3. Example Customer / Support Pairs
Demonstrates how multi-part Spotify tweets (split into 1: and 2: due to Twitter's character limit) were chronologically merged into unified, actionable support solutions.

In [ ]:
for i, row in df_cases.iloc[:3].iterrows():
    print(f"--- [{row['case_id']}] Customer Tweet ID: {row['customer_tweet_id']} | Thread: {row['conversation_id']} ---")
    print(f"CUSTOMER: {row['customer_text']}")
    print(f"SPOTIFY:  {row['support_text']}\n")

### 4. Semantic Embedding Model (`sentence-transformers/all-MiniLM-L6-v2`)
We use a lightweight, CPU-efficient sentence embedding model that maps customer tweets into a 384-dimensional dense semantic vector space.

In [ ]:
model = SentenceEmbeddingModel()
print(f"Embedding Model:        {model.model_name}")
print(f"Vector Dimensions:      {model.embedding_dim}")
print(f"Target Inference Unit:  CPU (Thread-Safe)")

### 5. Pairwise Semantic Cosine Similarity Demonstration
Compare four queries to demonstrate semantic clustering:
- `S0` and `S1` describe the exact same billing issue using completely different words ("charged twice" vs "deducted two times").
- `S2` describes a playback bug ("music pausing").
- `S3` describes an account recovery issue ("reset password").

In [ ]:
sample_sentences = [
    "I was charged twice for Spotify Premium subscription",
    "My bank statement shows money deducted two times for Spotify",
    "Music keeps pausing automatically on my iPhone whenever screen goes black",
    "I forgot my password and cannot log in to my account"
]

embeddings = model.encode(sample_sentences, normalize_embeddings=True)
similarity_matrix = np.dot(embeddings, embeddings.T)

fig, ax = plt.subplots(figsize=(7, 6))
cax = ax.matshow(similarity_matrix, cmap="Blues", vmin=0, vmax=1)
fig.colorbar(cax)

labels = [f"S{i}" for i in range(len(sample_sentences))]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

for i in range(len(sample_sentences)):
    for j in range(len(sample_sentences)):
        val = similarity_matrix[i, j]
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", color="white" if val > 0.5 else "black", fontweight="bold")

plt.title("Pairwise Cosine Similarity Matrix (all-MiniLM-L6-v2)", pad=20, fontweight="bold")
plt.tight_layout()
plt.show()

for i, s in enumerate(sample_sentences):
    print(f"S{i}: {s}")
print(f"\nSemantic similarity between S0 and S1 (lexically distinct duplicate billing): {similarity_matrix[0, 1]:.4f}")
print(f"Semantic similarity between S0 and S2 (billing vs playback bug):             {similarity_matrix[0, 2]:.4f}")

### 6. Interactive Top-K Historical Retrieval
We query the persisted FAISS vector index using `retrieve_similar_cases(query, top_k=5)`.

In [ ]:
query = "I was charged twice for Spotify Premium"
results = retrieve_similar_cases(query, top_k=5)

print(f"QUERY: '{query}'")
print(f"RETRIEVED {len(results)} HISTORICAL CASES:\n")

for r in results:
    print(f"[Rank {r['rank']}] Case ID: {r['case_id']} | Similarity: {r['similarity']:.4f}")
    print(f"  Customer: {r['customer_text']}")
    print(f"  Spotify:  {r['support_text']}\n")

### 7. Searching Across Different Support Domains

In [ ]:
test_queries = [
    "Offline downloaded music won't play without internet connection",
    "How do I reset my password if I don't receive the email?",
    "My Discover Weekly playlist did not refresh on Monday",
    "Can I merge two separate Spotify accounts into one?"
]

for q in test_queries:
    print(f"==================================================================")
    print(f"QUERY: {q}")
    print(f"==================================================================")
    top_case = retrieve_similar_cases(q, top_k=1)[0]
    print(f"Top Match: {top_case['case_id']} (Cosine Similarity: {top_case['similarity']:.4f})")
    print(f"Customer:  {top_case['customer_text']}")
    print(f"Spotify:   {top_case['support_text']}\n")

### 8. Qualitative Examples Across Archetypes
Inspecting the curated qualitative examples from `evaluation/results/retrieval_examples.csv`.

In [ ]:
examples_path = os.path.join(project_root, "evaluation", "results", "retrieval_examples.csv")
if os.path.exists(examples_path):
    ex_df = pd.read_csv(examples_path)
    archetypes = ex_df["archetype"].unique()
    for arch in archetypes:
        sub = ex_df[ex_df["archetype"] == arch]
        print(f"------------------------------------------------------------------")
        print(f"ARCHETYPE: {arch.upper()}")
        print(f"Description: {sub.iloc[0]['archetype_description']}")
        print(f"Query: '{sub.iloc[0]['query_text']}'")
        print(f"Top-1 Retrieved Similarity: {sub.iloc[0]['similarity']:.4f}")
        print(f"Historical Customer: {sub.iloc[0]['historical_customer_text'][:90]}...")
        print(f"Spotify Response:    {sub.iloc[0]['historical_support_text'][:100]}...")
else:
    print("Run 'python -m backend.src.retrieval.evaluate' to generate evaluation results.")

### 9. Retrieval Metrics Summary
Displays the empirical evaluation results from `evaluation/results/retrieval_results.csv`.

In [ ]:
results_path = os.path.join(project_root, "evaluation", "results", "retrieval_results.csv")
if os.path.exists(results_path):
    res_df = pd.read_csv(results_path)
    display(res_df)
else:
    print("Run 'python -m backend.src.retrieval.evaluate' to view complete results table.")